# Grouped Query Attention (GQA) & Multi-Query Attention (MQA)

Walkthrough for **ai-learn-28-grouped-query-attention**.

GQA shares each K/V head across a group of query heads (`n_rep = Hq / Hkv`). MQA is the extreme `Hkv = 1`. Both cut KV-cache size at decode time (Ainslie et al. 2023; Llama-2).

## 1. Repeat KV along the head axis

`repeat_kv` expands `(B, Hkv, T, Dh)` → `(B, Hq, T, Dh)` by repeating each KV head `n_rep` times.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import numpy as np
from gqa import repeat_kv, n_rep, kv_savings_table, mode_from_heads

print("n_rep(8,2) =", n_rep(8, 2), "mode", mode_from_heads(8, 2))
K = np.arange(2 * 2 * 3 * 2).reshape(2, 2, 3, 2).astype(float)  # B=2,Hkv=2,T=3,Dh=2
K_exp = repeat_kv(K, n_rep(8, 2))
print("in", K.shape, "out", K_exp.shape)
print("head0==head1 (same KV group)?", np.allclose(K_exp[0, 0], K_exp[0, 1]))

## 2. KV-cache bytes: MHA vs GQA vs MQA

Cache stores K and V for every token — scales with `n_kv_heads`, not `n_q_heads`.

In [ ]:
table = kv_savings_table(8, n_kv_gqa=2, batch=1, seq_len=2048, d_head=64)
for name, row in table["modes"].items():
    print(f"{name:3s} Hkv={row['n_kv_heads']}  MiB={row['mib']:.3f}  vs_MHA={row['vs_mha_ratio']}×")

## 3. Output fidelity vs full MHA

Collapse MHA's K/V within groups (mean) → GQA/MQA, expand, compare cosine of outputs.

In [ ]:
from experiments import random_weight_fidelity

fid = random_weight_fidelity(seed=42)
print("GQA cosine", fid["cosine_gqa_collapsed"])
print("MQA cosine", fid["cosine_mqa_collapsed"])
print("GQA closer?", fid["gqa_closer_than_mqa"])

## 4. Matching task after a short train

Planted-key retrieval: GQA should approach MHA; MQA may lag.

In [ ]:
from experiments import compare_modes_trained

r = compare_modes_trained(seed=42, steps=120)
for mode, row in r["by_mode"].items():
    print(f"{mode:3s} n_kv={row['n_kv_heads']} acc={row['final_acc']} mass={row['final_mass']}")

## 5. Full smoke

```bash
python run_smoke.py   # writes results/ (seed 42)
```

See `results/RESULTS.md` for tables and SVG plots.

### MHA vs GQA vs MQA
| | MHA | GQA | MQA |
|---|---|---|---|
| n_kv_heads | = Hq | divides Hq | 1 |
| KV cache | largest | Hq/Hkv × smaller | Hq × smaller |
| Capacity | full | mild share | aggressive share |
| Used in | classic Transformers | Llama-2/3, Mistral | PaLM-style MQA |